# Phase 5 — Machine Learning: Classification & Forecasting
### Social Services Demand Risk Predictor

---

This notebook trains and evaluates predictive models for two tasks:

**Task A — Classification:** Predict whether an SA2 region is High / Medium / Low 
risk for welfare demand growth using Logistic Regression, Random Forest, and XGBoost.

**Task B — Time-Series Forecasting:** Forecast quarterly DSS payment recipient 
counts 4 quarters ahead using ARIMA and Facebook Prophet.

**Inputs:**  
- `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`  
- `data/processed/y_train.csv`, `y_val.csv`, `y_test.csv`  
- `data/processed/dss_payments_clean.csv`  

**Outputs:**  
- `models/logistic_regression.joblib`  
- `models/random_forest.joblib`  
- `models/xgboost_model.joblib`  
- `models/prophet_model.joblib`  
- `reports/model_comparison.csv`  

**Libraries:** scikit-learn, xgboost, shap, prophet, statsmodels

## 1. Setup — Imports & Paths

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Classification models ──────────────────────────────────────────────────
from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing  import LabelEncoder
from sklearn.metrics        import (classification_report, confusion_matrix,
                                    f1_score, roc_auc_score, ConfusionMatrixDisplay,
                                    precision_recall_curve, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

# ── XGBoost ────────────────────────────────────────────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f"  xgboost      : {xgb.__version__}")
except ImportError:
    XGB_AVAILABLE = False
    print("  xgboost not installed — run: pip install xgboost")

# ── SHAP (explainability) ──────────────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
    print(f"  shap         : {shap.__version__}")
except ImportError:
    SHAP_AVAILABLE = False
    print("  shap not installed — run: pip install shap")

# ── Time-series ────────────────────────────────────────────────────────────
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print("  prophet      : available")
except ImportError:
    PROPHET_AVAILABLE = False
    print("  prophet not installed — run: pip install prophet")

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools   import adfuller
    STATSMODELS_AVAILABLE = True
    print("  statsmodels  : available")
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("  statsmodels not installed — run: pip install statsmodels")

# ── Serialisation ──────────────────────────────────────────────────────────
import joblib

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"       : 120,
    "figure.facecolor" : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

print()
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")
print(f"  scikit-learn : imported successfully")

In [ ]:
# Paths

NOTEBOOK_DIR  = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR    = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR   = os.path.join(PROJECT_ROOT, "reports")

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

RANDOM_STATE = 42   # Fixed seed — ensures reproducibility across all models

print("Paths configured:")
print(f"  Processed : {PROCESSED_DIR}")
print(f"  Models    : {MODELS_DIR}")
print(f"  Reports   : {REPORTS_DIR}")

## 2. Load Train / Validation / Test Splits

In [ ]:
def load_split(filename, label):
    path = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  [✓] {label:<20} {df.shape}")
        return df
    else:
        print(f"  [✗] {label} NOT FOUND — run Phase 4 first: {path}")
        return None

print("Loading splits from Phase 4...")
print("-" * 45)
X_train = load_split("X_train.csv", "X_train")
X_val   = load_split("X_val.csv",   "X_val")
X_test  = load_split("X_test.csv",  "X_test")
y_train_df = load_split("y_train.csv", "y_train")
y_val_df   = load_split("y_val.csv",   "y_val")
y_test_df  = load_split("y_test.csv",  "y_test")
print("-" * 45)

# Flatten y to 1D Series
if y_train_df is not None:
    y_train = y_train_df.iloc[:, 0]
    y_val   = y_val_df.iloc[:, 0]
    y_test  = y_test_df.iloc[:, 0]

    print()
    print("Target class distribution — train set:")
    for cls, n in y_train.value_counts().items():
        pct = n / len(y_train) * 100
        print(f"  {cls:<8} {n:>5,}  ({pct:.1f}%)")

In [ ]:
# Load selected feature list saved by Phase 4

feat_path = os.path.join(PROCESSED_DIR, "selected_features.txt")
if os.path.exists(feat_path):
    with open(feat_path) as f:
        selected_features = [line.strip() for line in f if line.strip()]
    print(f"Selected features ({len(selected_features)}):")
    for feat in selected_features:
        print(f"  {feat}")
else:
    # Fall back to whatever columns are in X_train
    selected_features = X_train.columns.tolist() if X_train is not None else []
    print(f"Feature file not found — using all {len(selected_features)} columns from X_train")

# Align all splits to selected features that exist
if X_train is not None:
    available = [f for f in selected_features if f in X_train.columns]
    X_train = X_train[available]
    X_val   = X_val[available]
    X_test  = X_test[available]
    print(f"\nFeature matrix shape: {X_train.shape}")

## 3. Helper Functions

We define reusable evaluation functions so every model is assessed 
the same way — consistent, fair comparison.

In [ ]:
def evaluate_classifier(model, X, y_true, label="Model", split="Validation"):
    """
    Evaluate a trained classifier and return a metrics dictionary.
    Prints a formatted report and returns metrics for comparison table.
    """
    y_pred = model.predict(X)

    # F1 macro treats every class equally — important for imbalanced data
    f1_macro = f1_score(y_true, y_pred, average="macro",  zero_division=0)
    f1_wtd   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    # ROC-AUC (one-vs-rest, macro) — needs probability scores
    try:
        y_prob   = model.predict_proba(X)
        le_tmp   = LabelEncoder()
        y_enc    = le_tmp.fit_transform(y_true)
        roc_auc  = roc_auc_score(y_enc, y_prob, multi_class="ovr", average="macro")
    except Exception:
        roc_auc  = np.nan

    print(f"\n{'='*55}")
    print(f"  {label} — {split} Set Results")
    print(f"{'='*55}")
    print(f"  F1 (macro)    : {f1_macro:.4f}")
    print(f"  F1 (weighted) : {f1_wtd:.4f}")
    print(f"  ROC-AUC       : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC      : N/A")
    print()
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "model"       : label,
        "split"       : split,
        "f1_macro"    : round(f1_macro, 4),
        "f1_weighted" : round(f1_wtd,   4),
        "roc_auc"     : round(roc_auc,  4) if not np.isnan(roc_auc) else None,
        "y_pred"      : y_pred
    }


def plot_confusion_matrix(model, X, y_true, label, filename):
    """Plot and save a styled confusion matrix."""
    y_pred  = model.predict(X)
    classes = sorted(y_true.unique())
    cm      = confusion_matrix(y_true, y_pred, labels=classes)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{label} — Confusion Matrix", fontweight="bold", fontsize=12)
    plt.tight_layout()
    path = os.path.join(REPORTS_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[✓] Saved: reports/{filename}")


print("Helper functions defined:")
print("  evaluate_classifier()")
print("  plot_confusion_matrix()")

## 4. Model A — Logistic Regression (Baseline)

Logistic Regression is always the first model to try. It is:
- Fast to train
- Highly interpretable (coefficients show feature direction)
- A strong baseline that more complex models must beat to justify their complexity

If Logistic Regression already achieves F1 > 0.70, we may not need XGBoost at all.

In [ ]:
if X_train is not None:
    print("Training Logistic Regression...")
    print("-" * 45)

    lr_model = LogisticRegression(
        C            = 1.0,       # Regularisation strength (1/lambda)
        max_iter     = 1000,      # Increase from default 100 — ensures convergence
        class_weight = "balanced", # Automatically adjusts for class imbalance
        random_state = RANDOM_STATE,
        solver       = "lbfgs"    # Efficient for multi-class problems
    )


    lr_model.fit(X_train, y_train)
    print("Training complete.")

    # Evaluate on validation set
    lr_results = evaluate_classifier(lr_model, X_val, y_val, "Logistic Regression")

In [ ]:
if X_train is not None:
    plot_confusion_matrix(lr_model, X_val, y_val,
                          "Logistic Regression", "13_lr_confusion_matrix.png")

In [ ]:
if X_train is not None:
    # Logistic Regression coefficients — shows direction of each feature's effect
    coef_df = pd.DataFrame(
        lr_model.coef_,
        columns = X_train.columns,
        index   = lr_model.classes_
    ).T

    print("Logistic Regression Coefficients (by class):")
    print("Positive = increases probability of that class")
    print("-" * 55)
    display(coef_df.round(3))

    # Plot coefficients for the 'High' risk class
    high_class = "High" if "High" in coef_df.columns else coef_df.columns[0]
    coef_sorted = coef_df[high_class].sort_values()

    fig, ax = plt.subplots(figsize=(8, 5))
    colors  = ["#993C1D" if v > 0 else "#1D9E75" for v in coef_sorted.values]
    ax.barh(coef_sorted.index, coef_sorted.values,
            color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Logistic Regression Coefficients\n(Predicting '{high_class}' risk class)",
                 fontweight="bold")
    ax.set_xlabel("Coefficient value")
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "14_lr_coefficients.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/14_lr_coefficients.png")

## 5. Model B — Random Forest

Random Forest builds many decision trees on random subsets of the data 
and averages their predictions. Key advantages:
- Handles non-linear relationships automatically
- Robust to outliers
- Provides built-in feature importance
- Less prone to overfitting than a single decision tree

We use cross-validation to tune the key hyperparameters.

In [ ]:
if X_train is not None:
    print("Training Random Forest...")
    print("-" * 45)

    # Start with sensible defaults — we will tune below
    rf_model = RandomForestClassifier(
        n_estimators  = 200,          # Number of trees — more is better up to a point
        max_depth     = None,         # Allow trees to grow fully — regularise via min_samples
        min_samples_split = 5,        # Minimum samples to split a node
        min_samples_leaf  = 2,        # Minimum samples in a leaf — prevents tiny leaves
        class_weight  = "balanced",   # Handle class imbalance
        random_state  = RANDOM_STATE,
        n_jobs        = -1            # Use all CPU cores
    )

    rf_model.fit(X_train, y_train)
    print("Training complete.")

    rf_results = evaluate_classifier(rf_model, X_val, y_val, "Random Forest")

In [ ]:
if X_train is not None:
    plot_confusion_matrix(rf_model, X_val, y_val,
                          "Random Forest", "15_rf_confusion_matrix.png")

In [ ]:
if X_train is not None:
    # Random Forest built-in feature importance (Gini impurity reduction)
    feat_imp = pd.Series(rf_model.feature_importances_,
                         index=X_train.columns).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    feat_imp.plot(kind="barh", color="#534AB7", edgecolor="white",
                  linewidth=0.5, ax=ax)
    ax.set_title("Random Forest — Feature Importance (Gini)",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("Mean decrease in impurity")
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "16_rf_feature_importance.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/16_rf_feature_importance.png")

    print("\nFeature importance ranking:")
    for feat, imp in feat_imp.sort_values(ascending=False).items():
        bar = "█" * int(imp * 100)
        print(f"  {feat:<30} {imp:.4f}  {bar}")

## 6. Hyperparameter Tuning — Random Forest

Grid Search with Stratified K-Fold cross-validation finds the best 
combination of hyperparameters. We use 5-fold CV so every sample is 
used for both training and validation across the folds.

In [ ]:
if X_train is not None:
    print("Running Grid Search CV on Random Forest...")
    print("This may take 2–5 minutes depending on dataset size.")
    print("-" * 50)

    param_grid = {
        "n_estimators"     : [100, 200],
        "max_depth"        : [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf" : [1, 2],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid_search = GridSearchCV(
        estimator  = RandomForestClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE,
                                             n_jobs=-1),
        param_grid = param_grid,
        cv         = cv,
        scoring    = "f1_macro",   # Optimise for macro F1
        n_jobs     = -1,
        verbose    = 1
    )

    grid_search.fit(X_train, y_train)

    print()
    print(f"Best parameters  : {grid_search.best_params_}")
    print(f"Best CV F1 macro : {grid_search.best_score_:.4f}")

    # Refit best model
    rf_best = grid_search.best_estimator_
    rf_tuned_results = evaluate_classifier(rf_best, X_val, y_val,
                                            "Random Forest (Tuned)")

## 7. Model C — XGBoost

XGBoost (Extreme Gradient Boosting) builds trees sequentially — each 
tree corrects the errors of the previous one. It consistently wins 
tabular data competitions and is widely used in Australian government 
analytics for its performance and interpretability via SHAP.

In [ ]:
if X_train is not None and XGB_AVAILABLE:
    print("Training XGBoost...")
    print("-" * 45)

    # Encode target labels to integers (XGBoost requires numeric labels)
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc   = le.transform(y_val)
    y_test_enc  = le.transform(y_test)

    print(f"Class mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    print()

    # Calculate class weights for imbalance handling
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight("balanced", y_train_enc)

    xgb_model = xgb.XGBClassifier(
        n_estimators       = 300,
        max_depth          = 6,
        learning_rate      = 0.05,    # Small learning rate + more trees = better generalisation
        subsample          = 0.8,     # Row sampling per tree — reduces overfitting
        colsample_bytree   = 0.8,     # Feature sampling per tree — reduces overfitting
        use_label_encoder  = False,
        eval_metric        = "mlogloss",
        random_state       = RANDOM_STATE,
        n_jobs             = -1,
        verbosity          = 0
    )

    xgb_model.fit(
        X_train, y_train_enc,
        sample_weight   = sample_weights,
        eval_set        = [(X_val, y_val_enc)],
        verbose         = False
    )

    print("Training complete.")

    # Decode predictions back to original class labels for evaluation
    xgb_pred_enc = xgb_model.predict(X_val)
    xgb_pred     = le.inverse_transform(xgb_pred_enc)

    # Manual evaluation (wrapper function needs string labels)
    xgb_f1_macro = f1_score(y_val, xgb_pred, average="macro",    zero_division=0)
    xgb_f1_wtd   = f1_score(y_val, xgb_pred, average="weighted", zero_division=0)

    print(f"\n{'='*55}")
    print(f"  XGBoost — Validation Set Results")
    print(f"{'='*55}")
    print(f"  F1 (macro)    : {xgb_f1_macro:.4f}")
    print(f"  F1 (weighted) : {xgb_f1_wtd:.4f}")
    print()
    print(classification_report(y_val, xgb_pred, zero_division=0))

    xgb_results = {
        "model"       : "XGBoost",
        "split"       : "Validation",
        "f1_macro"    : round(xgb_f1_macro, 4),
        "f1_weighted" : round(xgb_f1_wtd,   4),
        "y_pred"      : xgb_pred
    }

elif not XGB_AVAILABLE:
    print("XGBoost not installed. Run: pip install xgboost")

In [ ]:
if X_train is not None and XGB_AVAILABLE:
    # Confusion matrix for XGBoost
    classes  = sorted(y_val.unique())
    cm       = confusion_matrix(y_val, xgb_pred, labels=classes)
    fig, ax  = plt.subplots(figsize=(6, 5))
    disp     = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("XGBoost — Confusion Matrix", fontweight="bold", fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "17_xgb_confusion_matrix.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/17_xgb_confusion_matrix.png")

## 8. SHAP Explainability

SHAP (SHapley Additive exPlanations) answers the question:
*"Why did the model make this prediction?"*

It assigns each feature a contribution value for every individual 
prediction — not just a global importance score. This is critical 
for government DS work where decisions need to be explainable to 
non-technical stakeholders and must satisfy accountability requirements.

In [ ]:
if X_train is not None and SHAP_AVAILABLE and XGB_AVAILABLE:
    print("Computing SHAP values for XGBoost model...")
    print("(This may take 30–60 seconds)")
    print("-" * 45)

    # TreeExplainer is optimised for tree-based models (RF, XGBoost, LightGBM)
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_val)

    print("SHAP values computed.")
    print(f"  Shape: {np.array(shap_values).shape}")
    print()

    # Summary plot — shows global feature importance + direction of effect
    # Each dot is one SA2 region:
    #   x-axis = SHAP value (+ means pushes toward this class)
    #   colour = feature value (red = high, blue = low)
    plt.figure(figsize=(9, 6))
    shap.summary_plot(
        shap_values[0] if isinstance(shap_values, list) else shap_values,
        X_val,
        feature_names = X_val.columns.tolist(),
        plot_type     = "dot",
        show          = False
    )
    plt.title("SHAP Summary Plot — XGBoost\n(Impact of each feature on model predictions)",
              fontweight="bold", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, "18_shap_summary.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/18_shap_summary.png")

elif not SHAP_AVAILABLE:
    print("SHAP not installed. Run: pip install shap")

In [ ]:
if X_train is not None and SHAP_AVAILABLE and XGB_AVAILABLE:
    # SHAP bar plot — mean absolute SHAP value per feature (global importance)
    plt.figure(figsize=(8, 5))
    shap.summary_plot(
        shap_values[0] if isinstance(shap_values, list) else shap_values,
        X_val,
        feature_names = X_val.columns.tolist(),
        plot_type     = "bar",
        show          = False
    )
    plt.title("SHAP Feature Importance (Mean |SHAP value|)",
              fontweight="bold", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, "19_shap_bar.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/19_shap_bar.png")

In [ ]:
if X_train is not None and SHAP_AVAILABLE and XGB_AVAILABLE:
    # Waterfall plot for a single prediction — most useful for stakeholders
    # Shows exactly why one specific SA2 region was classified as High/Medium/Low

    sample_idx = 0   # Change this index to inspect a different SA2 region

    print(f"SHAP Waterfall — Individual Prediction Explanation")
    print(f"  SA2 region index : {sample_idx}")
    print(f"  True label       : {y_val.iloc[sample_idx]}")
    print(f"  Predicted label  : {xgb_pred[sample_idx]}")
    print()

    try:
        shap_exp = shap.Explanation(
            values    = explainer.shap_values(X_val.iloc[[sample_idx]])[0]
                        if isinstance(shap_values, list)
                        else explainer.shap_values(X_val.iloc[[sample_idx]]),
            base_values = explainer.expected_value[0]
                          if isinstance(explainer.expected_value, (list, np.ndarray))
                          else explainer.expected_value,
            data        = X_val.iloc[sample_idx].values,
            feature_names = X_val.columns.tolist()
        )
        plt.figure(figsize=(9, 5))
        shap.plots.waterfall(shap_exp, show=False)
        plt.title(f"Individual Prediction Explanation (SA2 index {sample_idx})",
                  fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(REPORTS_DIR, "20_shap_waterfall.png"),
                    dpi=150, bbox_inches="tight")
        plt.show()
        print("[✓] Saved: reports/20_shap_waterfall.png")
    except Exception as e:
        print(f"  Waterfall plot skipped: {e}")

## 9. Model Comparison

We compare all three classifiers side-by-side on the validation set 
to select the best model for final evaluation on the test set.

In [ ]:
if X_train is not None:
    # Build comparison table
    comparison_rows = []

    # Logistic Regression
    comparison_rows.append({
        "Model"        : "Logistic Regression",
        "F1 Macro"     : lr_results["f1_macro"],
        "F1 Weighted"  : lr_results["f1_weighted"],
        "ROC-AUC"      : lr_results.get("roc_auc", None),
        "Interpretable": "Yes — coefficients",
        "Training Speed": "Fast"
    })

    # Random Forest Tuned
    comparison_rows.append({
        "Model"        : "Random Forest (Tuned)",
        "F1 Macro"     : rf_tuned_results["f1_macro"],
        "F1 Weighted"  : rf_tuned_results["f1_weighted"],
        "ROC-AUC"      : rf_tuned_results.get("roc_auc", None),
        "Interpretable": "Partial — feature importance",
        "Training Speed": "Medium"
    })

    # XGBoost
    if XGB_AVAILABLE:
        comparison_rows.append({
            "Model"        : "XGBoost",
            "F1 Macro"     : xgb_results["f1_macro"],
            "F1 Weighted"  : xgb_results["f1_weighted"],
            "ROC-AUC"      : xgb_results.get("roc_auc", None),
            "Interpretable": "Yes — via SHAP",
            "Training Speed": "Fast"
        })

    df_comparison = pd.DataFrame(comparison_rows)
    df_comparison = df_comparison.sort_values("F1 Macro", ascending=False).reset_index(drop=True)

    print("Model Comparison — Validation Set")
    print("=" * 65)
    display(df_comparison)

    best_model_name = df_comparison.iloc[0]["Model"]
    best_f1         = df_comparison.iloc[0]["F1 Macro"]
    print(f"\nBest model: {best_model_name}  (F1 macro = {best_f1:.4f})")

    # Save comparison table
    comp_path = os.path.join(REPORTS_DIR, "model_comparison.csv")
    df_comparison.to_csv(comp_path, index=False)
    print(f"[✓] Saved: reports/model_comparison.csv")

In [ ]:
if X_train is not None:
    # Visual model comparison bar chart
    fig, ax = plt.subplots(figsize=(9, 5))

    x     = np.arange(len(df_comparison))
    width = 0.35

    bars1 = ax.bar(x - width/2, df_comparison["F1 Macro"],    width,
                   label="F1 Macro",    color="#1D9E75", edgecolor="white")
    bars2 = ax.bar(x + width/2, df_comparison["F1 Weighted"], width,
                   label="F1 Weighted", color="#534AB7", edgecolor="white")

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

    ax.axhline(0.70, color="#993C1D", linestyle="--", linewidth=1.2,
               label="Target (F1 ≥ 0.70)")
    ax.set_xticks(x)
    ax.set_xticklabels(df_comparison["Model"], rotation=10, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.set_title("Model Comparison — F1 Scores on Validation Set",
                 fontweight="bold", fontsize=13)
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "21_model_comparison.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/21_model_comparison.png")

## 10. Final Evaluation on Test Set

We evaluate the best model on the **test set exactly once**. 
The test set has never been seen during training or tuning — 
this gives an unbiased estimate of real-world performance.

**Important:** If you are unhappy with the test set result, you 
cannot go back and retune — that would invalidate the test set. 
The correct approach is to document the result honestly.

In [ ]:
if X_train is not None:
    # Select best model object based on comparison table
    model_map = {
        "Logistic Regression"  : lr_model,
        "Random Forest (Tuned)": rf_best,
    }
    if XGB_AVAILABLE:
        model_map["XGBoost"] = None   # XGBoost needs special handling below

    best_name = df_comparison.iloc[0]["Model"]
    print(f"Best model selected: {best_name}")
    print("Running final evaluation on held-out test set...")
    print("=" * 55)

    if best_name == "XGBoost" and XGB_AVAILABLE:
        # XGBoost: encode + predict + decode
        y_test_pred_enc = xgb_model.predict(X_test)
        y_test_pred     = le.inverse_transform(y_test_pred_enc)

        test_f1_macro = f1_score(y_test, y_test_pred, average="macro",    zero_division=0)
        test_f1_wtd   = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)

        print(f"  F1 (macro)    : {test_f1_macro:.4f}")
        print(f"  F1 (weighted) : {test_f1_wtd:.4f}")
        print()
        print(classification_report(y_test, y_test_pred, zero_division=0))

        best_model_obj = xgb_model
        y_test_pred_final = y_test_pred

    else:
        best_model_obj = model_map[best_name]
        test_results   = evaluate_classifier(best_model_obj, X_test, y_test,
                                              best_name, split="TEST (Final)")
        y_test_pred_final = test_results["y_pred"]

In [ ]:
if X_train is not None:
    # Final confusion matrix on test set
    classes  = sorted(y_test.unique())
    cm       = confusion_matrix(y_test, y_test_pred_final, labels=classes)
    fig, ax  = plt.subplots(figsize=(6, 5))
    disp     = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(f"{best_name}\nFinal Test Set Confusion Matrix",
                 fontweight="bold", fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "22_final_test_confusion_matrix.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/22_final_test_confusion_matrix.png")

## 11. Save All Models

In [ ]:
if X_train is not None:
    saves = [
        (lr_model,  "logistic_regression.joblib"),
        (rf_best,   "random_forest.joblib"),
    ]
    if XGB_AVAILABLE:
        saves.append((xgb_model, "xgboost_model.joblib"))

    print("Saving models to models/ directory...")
    print("-" * 45)
    for model_obj, fname in saves:
        path = os.path.join(MODELS_DIR, fname)
        joblib.dump(model_obj, path)
        size_kb = os.path.getsize(path) / 1024
        print(f"  [✓] {fname:<40} {size_kb:>8.1f} KB")

## 12. Task B — Time-Series Forecasting

We now switch to the forecasting task: predicting quarterly DSS payment 
recipient counts 4 quarters ahead. We use two approaches:

1. **ARIMA** — classical statistical model, good for stable trends
2. **Prophet** — Facebook's forecasting library, handles seasonality 
   and holidays automatically, very popular in government analytics

In [ ]:
# Load DSS payment data for time-series analysis

dss_path = os.path.join(PROCESSED_DIR, "dss_payments_clean.csv")

if os.path.exists(dss_path):
    df_dss = pd.read_csv(dss_path)
    print(f"DSS data loaded: {df_dss.shape}")
    print()
    print("Columns:")
    for c in df_dss.columns:
        print(f"  {c}")
else:
    print(f"DSS file not found at {dss_path}")
    print("Run Phase 2 to generate it.")
    df_dss = None

In [ ]:
if df_dss is not None:
    # Build a national quarterly time-series of total DSS recipients
    # This aggregates all payment types and all SA2 regions into one national series

    # Find date and count columns
    date_col  = next((c for c in df_dss.columns
                      if any(kw in c.lower() for kw in ["date", "quarter", "period", "month"])), None)
    count_col = next((c for c in df_dss.columns
                      if any(kw in c.lower() for kw in ["recipient", "number", "count"])), None)

    print(f"Date column  identified : {date_col}")
    print(f"Count column identified : {count_col}")

    if date_col and count_col:
        df_dss[count_col] = pd.to_numeric(
            df_dss[count_col].astype(str).str.replace(",", ""), errors="coerce"
        )

        ts = (df_dss.groupby(date_col)[count_col]
                    .sum()
                    .reset_index()
                    .rename(columns={date_col: "ds", count_col: "y"}))

        # Parse dates
        ts["ds"] = pd.to_datetime(ts["ds"], infer_datetime_format=True, errors="coerce")
        ts = ts.dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True)

        print(f"\nTime-series built: {len(ts)} quarters")
        print(f"  From : {ts['ds'].min().date()}")
        print(f"  To   : {ts['ds'].max().date()}")
        print(f"  Total recipients range: {ts['y'].min():,.0f} — {ts['y'].max():,.0f}")
        display(ts.tail(8))
    else:
        print("\nCould not identify date or count column.")
        print("Creating synthetic quarterly series for demonstration...")
        dates = pd.date_range("2015-03-01", periods=36, freq="QS")
        base  = 800000
        trend = np.linspace(0, 150000, 36)
        seas  = 20000 * np.sin(np.linspace(0, 4 * np.pi, 36))
        noise = np.random.normal(0, 8000, 36)
        ts    = pd.DataFrame({"ds": dates, "y": base + trend + seas + noise})
        print(f"Synthetic series created: {len(ts)} quarters")

In [ ]:
if df_dss is not None and 'ts' in dir():
    # Plot the raw time-series
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(ts["ds"], ts["y"], color="#1D9E75", linewidth=2, marker="o",
            markersize=4, label="Actual recipients")
    ax.fill_between(ts["ds"], ts["y"], alpha=0.12, color="#1D9E75")
    ax.set_title("National DSS Payment Recipients — Quarterly Time Series",
                 fontweight="bold", fontsize=13)
    ax.set_xlabel("Quarter")
    ax.set_ylabel("Total Recipients")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "23_dss_time_series.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/23_dss_time_series.png")

In [ ]:
if 'ts' in dir() and STATSMODELS_AVAILABLE:
    series = ts["y"].dropna()

    result = adfuller(series, autolag="AIC")

    print("Augmented Dickey-Fuller Test — Stationarity Check")
    print("=" * 50)
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  Lags used     : {result[2]}")
    print()
    print("  Critical values:")
    for key, val in result[4].items():
        print(f"    {key} : {val:.4f}")

    print()
    if result[1] < 0.05:
        print("  ✓ Series is STATIONARY (p < 0.05) — use d=0 in ARIMA")
        arima_d = 0
    else:
        print("  ⚠ Series is NON-STATIONARY (p ≥ 0.05) — use d=1 in ARIMA")
        arima_d = 1

### 12b. ARIMA Forecast

In [ ]:
if 'ts' in dir() and STATSMODELS_AVAILABLE:
    FORECAST_PERIODS = 4   # Forecast 4 quarters ahead

    # Hold out last 4 quarters as test set for evaluation
    train_ts = ts[:-FORECAST_PERIODS]
    test_ts  = ts[-FORECAST_PERIODS:]

    print(f"ARIMA training series : {len(train_ts)} quarters")
    print(f"ARIMA test set        : {len(test_ts)} quarters (held out)")
    print()

    try:
        # ARIMA(p, d, q):
        # p = autoregressive order (look back 2 quarters)
        # d = differencing order (from stationarity test above)
        # q = moving average order
        arima = ARIMA(train_ts["y"], order=(2, arima_d, 1))
        arima_fit = arima.fit()

        print(arima_fit.summary())

        # Forecast
        forecast_result = arima_fit.forecast(steps=FORECAST_PERIODS)
        forecast_dates  = pd.date_range(
            start=train_ts["ds"].iloc[-1],
            periods=FORECAST_PERIODS + 1,
            freq="QS"
        )[1:]

        # MAPE on test set
        actuals  = test_ts["y"].values
        forecast = forecast_result.values
        mape     = np.mean(np.abs((actuals - forecast) / actuals)) * 100

        print(f"\nARIMA Forecast Results:")
        print(f"  MAPE on {FORECAST_PERIODS}-quarter hold-out: {mape:.2f}%")
        print()
        for d, f, a in zip(forecast_dates, forecast, actuals):
            print(f"  {d.date()}  Forecast: {f:>12,.0f}  Actual: {a:>12,.0f}  "
                  f"Error: {abs(f-a)/a*100:.1f}%")

    except Exception as e:
        print(f"ARIMA fitting error: {e}")
        print("Try adjusting the order parameters or check series length.")

### 12c. Prophet Forecast

In [ ]:
if 'ts' in dir() and PROPHET_AVAILABLE:
    FORECAST_PERIODS = 4

    train_prophet = ts[:-FORECAST_PERIODS].copy()
    test_prophet  = ts[-FORECAST_PERIODS:].copy()

    print("Training Prophet model...")
    print("-" * 45)

    prophet_model = Prophet(
        yearly_seasonality  = True,
        weekly_seasonality  = False,  # Quarterly data has no weekly pattern
        daily_seasonality   = False,
        seasonality_mode    = "additive",
        changepoint_prior_scale = 0.05,   # Controls trend flexibility
        interval_width      = 0.95        # 95% confidence intervals
    )

    # Add Australian public holidays effect
    # (impacts service demand patterns significantly)
    prophet_model.add_country_holidays(country_name="AU")

    prophet_model.fit(train_prophet[["ds", "y"]])
    print("Prophet training complete.")

    # Create future dataframe for forecast period
    future = prophet_model.make_future_dataframe(
        periods=FORECAST_PERIODS, freq="QS"
    )

    forecast_df = prophet_model.predict(future)

    # Evaluate on hold-out test quarters
    test_forecast = forecast_df.tail(FORECAST_PERIODS)
    actuals       = test_prophet["y"].values
    predictions   = test_forecast["yhat"].values

    prophet_mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"\nProphet Forecast Results:")
    print(f"  MAPE on {FORECAST_PERIODS}-quarter hold-out: {prophet_mape:.2f}%")
    print()
    for i in range(FORECAST_PERIODS):
        d   = test_forecast["ds"].iloc[i].date()
        yh  = test_forecast["yhat"].iloc[i]
        ylo = test_forecast["yhat_lower"].iloc[i]
        yhi = test_forecast["yhat_upper"].iloc[i]
        act = actuals[i]
        print(f"  {d}  Forecast: {yh:>12,.0f}  [{ylo:,.0f} — {yhi:,.0f}]  "
              f"Actual: {act:>12,.0f}  Error: {abs(yh-act)/act*100:.1f}%")

elif not PROPHET_AVAILABLE:
    print("Prophet not installed. Run: pip install prophet")

In [ ]:
if 'ts' in dir() and PROPHET_AVAILABLE:
    # Prophet forecast plot
    fig = prophet_model.plot(forecast_df, figsize=(12, 6))
    axes = fig.get_axes()
    axes[0].set_title("Prophet Forecast — National DSS Recipients (4 Quarters Ahead)",
                       fontweight="bold", fontsize=13)
    axes[0].set_ylabel("Total Recipients")
    axes[0].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    )
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "24_prophet_forecast.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/24_prophet_forecast.png")

In [ ]:
if 'ts' in dir() and PROPHET_AVAILABLE:
    # Prophet components plot — shows trend + seasonality separately
    fig2 = prophet_model.plot_components(forecast_df, figsize=(12, 8))
    plt.suptitle("Prophet Components — Trend & Seasonality Breakdown",
                 fontweight="bold", fontsize=13, y=1.01)
    plt.tight_layout()
    fig2.savefig(os.path.join(REPORTS_DIR, "25_prophet_components.png"),
                 dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/25_prophet_components.png")

In [ ]:
# Forecasting model comparison

if 'ts' in dir():
    print("Forecasting Model Comparison")
    print("=" * 50)
    print(f"  Target metric : MAPE ≤ 10%")
    print()

    rows = []
    if STATSMODELS_AVAILABLE and 'mape' in dir():
        rows.append({"Model": "ARIMA(2,d,1)", "MAPE (%)": round(mape, 2),
                     "Target Met": "✓" if mape <= 10 else "✗"})
    if PROPHET_AVAILABLE and 'prophet_mape' in dir():
        rows.append({"Model": "Prophet", "MAPE (%)": round(prophet_mape, 2),
                     "Target Met": "✓" if prophet_mape <= 10 else "✗"})

    if rows:
        df_fc = pd.DataFrame(rows).sort_values("MAPE (%)")
        display(df_fc)

        # Save Prophet model
        if PROPHET_AVAILABLE and 'prophet_model' in dir():
            prophet_path = os.path.join(MODELS_DIR, "prophet_model.joblib")
            joblib.dump(prophet_model, prophet_path)
            print(f"\n[✓] Prophet model saved → {prophet_path}")

## 13. Phase 5 Summary

In [ ]:
print("=" * 60)
print("  PHASE 5 COMPLETE — Machine Learning")
print("=" * 60)
print()

tasks = [
    ("TASK A — CLASSIFICATION", [
        "Logistic Regression baseline trained and evaluated",
        "Random Forest trained + Grid Search CV hyperparameter tuning",
        "XGBoost trained with class weights",
        "SHAP explainability — summary, bar, and waterfall plots",
        "Model comparison table saved to reports/model_comparison.csv",
        "Best model evaluated on held-out test set (one time only)",
        "All 3 models serialised to models/ directory",
    ]),
    ("TASK B — FORECASTING", [
        "Stationarity tested with Augmented Dickey-Fuller test",
        "ARIMA model fitted and evaluated on hold-out quarters",
        "Prophet model with Australian public holidays fitted",
        "4-quarter ahead forecast with 95% confidence intervals",
        "Forecast components plot (trend + seasonality breakdown)",
        "Both models evaluated against 10% MAPE target",
    ]),
]

for task_name, steps in tasks:
    print(f"  {task_name}")
    for step in steps:
        print(f"    ✓  {step}")
    print()

print("  Reports saved:")
reports = [
    "13_lr_confusion_matrix.png",   "14_lr_coefficients.png",
    "15_rf_confusion_matrix.png",   "16_rf_feature_importance.png",
    "17_xgb_confusion_matrix.png",  "18_shap_summary.png",
    "19_shap_bar.png",              "20_shap_waterfall.png",
    "21_model_comparison.png",      "22_final_test_confusion_matrix.png",
    "23_dss_time_series.png",       "24_prophet_forecast.png",
    "25_prophet_components.png",    "model_comparison.csv",
]
for r in reports:
    print(f"    reports/{r}")

print()
print("  Next notebook: 06_nlp_policy_analysis.ipynb")
print("=" * 60)

## 14. Phase 5 Complete

**What this notebook produced:**

**Classification (Task A):**
- ✓ Logistic Regression — interpretable baseline with coefficient plot
- ✓ Random Forest — tuned via Grid Search CV with feature importance
- ✓ XGBoost — gradient boosting with SHAP explainability (3 SHAP plots)
- ✓ Model comparison table — best model selected transparently
- ✓ Final test set evaluation — unbiased performance estimate
- ✓ All models saved to `models/`

**Forecasting (Task B):**
- ✓ Stationarity tested — ARIMA order determined from data
- ✓ ARIMA fitted and evaluated with MAPE
- ✓ Prophet fitted with Australian holidays + seasonality
- ✓ 4-quarter ahead forecast with confidence intervals
- ✓ Components plot showing trend and seasonal patterns

**Next:** `06_nlp_policy_analysis.ipynb` — extract and analyse text from 
DSS annual reports using topic modelling (LDA) and sentiment analysis.